# Sumarização e avaliação de desempenho de diálogos de atendimento ao cliente pelo Twitter

- Aplica modelos disponiblizados na Hugging Face para sumarização de diálogos no Twitter

- Comparar performance dos modelos para sumarização
  

## Configurações iniciais

In [ ]:
import os

from rich import print
import tqdm
from typing import List

from datasets import Dataset
import evaluate

import pandas as pd
from langchain_community.llms import HuggingFacePipeline

import pandas as pd
from tqdm import tqdm
import numpy as np
from rich import print

from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    AutoModelForSeq2SeqLM,
    pipeline
)
import torch

from IPython.display import Markdown, display

In [ ]:
LLM_MODEL_PARAMS = {
    'qwen_instruct': {
        'model_name': 'Qwen/Qwen2.5-0.5B-Instruct',
        'auto_model_class': AutoModelForCausalLM,
        'task_pipeline': 'text-generation'
    },
    't5_small': {
        'model_name': 'google-t5/t5-small',
        'auto_model_class': AutoModelForSeq2SeqLM,
        'task_pipeline': 'text-generation'
    },
    'falconsai': {
        'model_name': 'Falconsai/text_summarization',
        'auto_model_class': AutoModelForSeq2SeqLM,
        'task_pipeline': 'text-generation'
    }
}

# =============================
# MODEL_NAME = 'falconsai'
# MODEL_NAME = 't5_small'
MODEL_NAME = 'qwen_instruct'


N_MAX_SAMPLES_EVAL = 2327
# =============================


IS_CAUSAL = True if MODEL_NAME in ['qwen_instruct', 'gpt'] else False
LLM_MODEL_NAME = LLM_MODEL_PARAMS[MODEL_NAME]['model_name']
TASK_PIPLINE = LLM_MODEL_PARAMS[MODEL_NAME]['task_pipeline']
AUTO_MODEL_CLASS = LLM_MODEL_PARAMS[MODEL_NAME]['auto_model_class']

DEVICE = 'cuda'

PRED_COL_NAME = 'summary_pred'

PATH_PREPARED_DATASET = f'../data/interim/dataset_summarization.csv'
MODEL_PATH = r'../models'

In [4]:
df_tweets_summ = pd.read_csv(PATH_PREPARED_DATASET)
print(df_tweets_summ.shape)
df_tweets_summ.head()

(2327, 6)

,conversation_id,tweet_ids,created_at_list,tweet_texts,human_summary,elapsed_time
0,b065262210783596c1fe79466b8f8985,"[87073, 87074, 87076]","{87073: Timestamp('2017-11-28 21:57:00+0000', ...",So neither my iPhone nor my Apple Watch are re...,Customer enquired about his Iphone and Apple w...,0 days 01:11:08
1,b065262210783596c1fe79466b8f8985,"[87074, 87076]","{87074: Timestamp('2017-11-28 21:11:57+0000', ...",So neither my iPhone nor my Apple Watch are re...,The customer has a problem. The agent in a ver...,0 days 00:26:05
2,b065262210783596c1fe79466b8f8985,"[87068, 87069, 87072, 87076]","{87068: Timestamp('2017-11-30 14:29:00+0000', ...",So neither my iPhone nor my Apple Watch are re...,Health and activity functions are not working ...,1 days 17:43:08
3,1e1d8fd4f95c984fb78687c9e946dc97,"[607293, 607296, 607297, 607297]","{607293: Timestamp('2017-11-22 12:16:27+0000',...",@115850 hi team! i m planning to get Apple Air...,Customer is eager to know about the replacemen...,0 days 00:56:46
4,b5773077fa55c381260390472deff4c2,"[739293, 739295]","{739293: Timestamp('2017-10-18 15:22:17+0000',...",@AskAmex Signed up for new card with Delta to ...,Signed up for an AmexCard with Delta but it di...,0 days 00:16:02


### Checa amostras

In [5]:
idx_test = 30
print(f'Dialógo Twitter:\n{df_tweets_summ.iloc[idx_test].tweet_texts}')
print(f'Sumarização:\n{df_tweets_summ.iloc[idx_test].human_summary}')

Dialógo Twitter:
@AskeBay My sisters account has been locked as it has been linked to an account with a negative balance though this
is not possible!!! @124579 Calling in will be the fastest way to resolve any issues or if she has a Facebook acct 
she can reach out to us there. Thanks! M @124579 2/2 Facebook by visiting https://t.co/6n0U95WGZY or she can email 
us at __email__. We'll be happy to talk with her. ^CR

Sumarização:
Customer is asking about the account which is locked. Agent stated that this can be resolved by calling or by email
immediately.

In [6]:
def run_human_summary(df: pd.DataFrame = None) -> str:
    res = []
    for _, df_i in df.iterrows():
        res.append(df_i['human_summary'])
    return res

In [7]:
print(run_human_summary(df=df_tweets_summ.head()))

[
    'Customer enquired about his Iphone and Apple watch which is not showing his any steps/activity and health 
activities. Agent is asking to move to DM and look into it.',
    'The customer has a problem. The agent in a very professional way tries to help the client.',
    'Health and activity functions are not working with the smartwatch and phone. Asks if the customer had 
restarted the items, offers to take this to DM to help resolve the issue.',
    'Customer is eager to know about the replacement policy on the earphones he wishes to buy. Agent stated that it
only applies if the received item is defective or damaged.',
    "Signed up for an AmexCard with Delta but it didn't go through. Told to phone the new accounts team."
]

### Define modelo LLM

In [ ]:
def llm_summarization_causal(
    llm_model: HuggingFacePipeline,
    input_text: str
) -> str:
    '''Sumariza texto do input em apenas uma sentença.'''

    prompt = f'''
    You are a helpful assistant that summarizes text.

    Summarize the text below in *exactly one* sentence.

    Rules:
    - Preserve the original sentiment
    - Be concise
    - Output *only* the sentence

    Text:
    {input_text}

    One sentence summary:
    '''

    res = llm_model.invoke(prompt)

    res = res.strip().split('\n')[0]

    return res

In [ ]:
local_model_path = f'{MODEL_PATH}/{LLM_MODEL_NAME}'

tokenizer = AutoTokenizer.from_pretrained(local_model_path)
model = AUTO_MODEL_CLASS.from_pretrained(
    local_model_path,
    device_map="auto"
)

Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

In [ ]:
if IS_CAUSAL:
    pipe = pipeline(
        task=TASK_PIPLINE,
        model=model,
        tokenizer=tokenizer,
    )
    llm_model = HuggingFacePipeline(pipeline=pipe)
else:
    pass


/tmp/ipykernel_5081/3898380202.py:13: LangChainDeprecationWarning: The class `HuggingFacePipeline` was deprecated in LangChain 0.0.37 and will be removed in 1.0. An updated version of the class exists in the `langchain-huggingface package and should be used instead. To use it run `pip install -U `langchain-huggingface` and import as `from `langchain_huggingface import HuggingFacePipeline``.
  llm_model = HuggingFacePipeline(pipeline=pipe)


In [11]:
df_eval = df_tweets_summ.head(N_MAX_SAMPLES_EVAL)
df_eval.rename(columns={'tweet_texts': 'inputs'}, inplace=True)
print(df_eval.shape)
df_eval.head(1)

(2327, 6)

,conversation_id,tweet_ids,created_at_list,inputs,human_summary,elapsed_time
0,b065262210783596c1fe79466b8f8985,"[87073, 87074, 87076]","{87073: Timestamp('2017-11-28 21:57:00+0000', ...",So neither my iPhone nor my Apple Watch are re...,Customer enquired about his Iphone and Apple w...,0 days 01:11:08


In [ ]:
def llm_summarization(input_text: str, model=model, tokenizer=tokenizer) -> str:
    prompt = f'summarize: {input_text}'

    inputs = tokenizer(
        prompt,
        return_tensors='pt',
        truncation=True
    ).to(model.device)

    outputs = model.generate(
        inputs['input_ids'], 
        do_sample=False,
    )

    return tokenizer.decode(outputs[0], skip_special_tokens=True).strip()

In [13]:
def run_llm_summarization(
    df: pd.DataFrame,
    is_causal: bool,
    llm_tokenizer=tokenizer,
    llm_model: HuggingFacePipeline = None,
) -> List[str]:
    '''Aplica sumarização em um conjunto de linhas.'''

    res = []
    for _, df_i in tqdm(df.iterrows(), total=len(df)):
        if is_causal:
            res_llm = llm_summarization_causal(
                llm_model=llm_model,
                input_text=df_i.inputs
            )
        else:
            res_llm = llm_summarization(
                input_text=df_i.inputs
            )
        res.append(res_llm)
    return res

In [14]:
if IS_CAUSAL:
    res_model = run_llm_summarization(
        llm_model=llm_model,
        df=df_eval.head(N_MAX_SAMPLES_EVAL),
        is_causal=IS_CAUSAL
    )
else:
    res_model = run_llm_summarization(
        df=df_eval.head(N_MAX_SAMPLES_EVAL),
        is_causal=IS_CAUSAL
    )
print(f'🤖 {LLM_MODEL_NAME}:\n\n{res_model[:3]} ...\n')

100%|██████████| 2327/2327 [1:51:14<00:00,  2.87s/it]


🤖 Qwen/Qwen2.5-0.5B-Instruct:

["The user's iPhone and Apple Watch are not recording their step activity or activities, and Health is unable to 
recognize either source due to an unknown issue. They would benefit from investigating together with 
@AppleSupport.", 'The user is unable to record their step activity or activity on their Apple Watch using either 
device, and they have no idea why. They want to investigate further together.', 'The user is experiencing issues 
with their iPhone and Apple Watch not recording their step data, and they have been unable to identify the cause of
these problems. They have also attempted to restart both devices but have found no resolution.'] ...

In [16]:
df_eval[PRED_COL_NAME] = res_model

In [17]:
model_name_output = LLM_MODEL_NAME.split('/')[0].lower()
output_path = f'../data/processed/df_eval_{model_name_output}.csv'

df_eval.to_csv(output_path, index=False)

print(f'CSV exportado: {output_path}')

CSV exportado: ../data/processed/df_eval_qwen.csv

### Checa sumarização

In [18]:
n_test = 7

print(df_eval[['inputs', PRED_COL_NAME]].iloc[n_test].to_dict())

{
    'inputs': "@AmazonHelp @115821 Wow, expected 4 packages yesterday, but only 2 showed up. 50% failure rate-not 
impressed. Glad I paid for fast shipping. @258930 I'm sorry you only received two of the orders. Is this happening 
with the same carrier each time? We can see what options are available for the lost items, reach us by phone or 
chat here: https://t.co/hApLpMlfHN ^MG",
    'summary_pred': 'The customer is disappointed because they did not receive all the packages and were not 
satisfied with the delivery service.'
}

## Calcula métricas

In [19]:
print(df_eval.shape)
df_eval.head(1)

(2327, 7)

,conversation_id,tweet_ids,created_at_list,inputs,human_summary,elapsed_time,summary_pred
0,b065262210783596c1fe79466b8f8985,"[87073, 87074, 87076]","{87073: Timestamp('2017-11-28 21:57:00+0000', ...",So neither my iPhone nor my Apple Watch are re...,Customer enquired about his Iphone and Apple w...,0 days 01:11:08,The user's iPhone and Apple Watch are not reco...


### Métrica [ROUGE (Recall-Oriented Understudy for Gisting Evaluation)](https://en.wikipedia.org/wiki/ROUGE_(metric))

In [20]:
def calc_rouge_metric(
    df: pd.DataFrame,
    target_name: str = 'target',
    pred_name: str = 'pred'
) -> pd.DataFrame:
    rouge_metric = evaluate.load('rouge')

    df_hf = Dataset.from_pandas(df)

    score = rouge_metric.compute(
        predictions=df_hf[target_name],
        references=df_hf[pred_name]
    )

    df_metrics = pd.DataFrame({LLM_MODEL_NAME: score})
    return df_metrics

In [21]:
df_metrics = calc_rouge_metric(df=df_eval, target_name='human_summary', pred_name=PRED_COL_NAME)
df_metrics = df_metrics.T.reset_index().rename(columns={'index': 'model'})
df_metrics

,model,rouge1,rouge2,rougeL,rougeLsum
0,Qwen/Qwen2.5-0.5B-Instruct,0.228629,0.049508,0.17128,0.171325


### Exporta métricas em CSV

In [ ]:
model_name_output = LLM_MODEL_NAME.split('/')[0].lower().replace('-', '_')
output_path = f'../data/processed/df_metrics_{model_name_output}.csv'

df_metrics.to_csv(output_path, index=False)

print(f'CSV exportado: {output_path}')

CSV exportado: ../data/processed/df_metrics_qwen.csv